In [ ]:
from wsd.WordSenseDisambiguator import WordSenseDisambiguator

In [ ]:
wsd = WordSenseDisambiguator('auto_training_data_wsd')

In [ ]:
gold_synset, gold_certainty, gold_lemmas = wsd.read_gold_data('wsd/wsd_gold_data_greek.tsv')

In [ ]:
# training_data gives for each lemma a mapping of glaux_id: synset (cluster) for each training instance (excluding instances that are in the gold data)
# lemma_synset_cluster gives for each clustered synset the name of the cluster that the synset is part of
training_data, lemma_synset_cluster = wsd.get_training_instances(set(gold_lemmas.values()),gold_synset)

In [ ]:
# test_data_norm has the same structure as training_data
test_data_norm = wsd.get_test_instances(gold_synset,gold_lemmas,lemma_synset_cluster)

In [ ]:
# Since part of GLAUx is copy-right restricted, I unfortunately cannot provide the full dataset before embeddings are extracted from it
# Instead the embeddings are read from a file
# They were extracted with the following code (VectorExtractor is from glaux-nlp):
# extractor = VectorExtractor(transformer_path='glaux-nlp/electra-grc',layers=[11],exclude_labels='_')
# dataset = extractor.build_dataset(wids,tokens,labels=labels,batched=True,batch_size=100,normalization_rule='greek_glaux')
# vectors = extractor.extract_vectors(dataset)
# For GreBERTa: transformer_path='bowphs/greberta' + tokenizer_add_prefix_space=True and normalization_rule = 'NFC'
from tqdm import tqdm
all_predictions = {}
for lemma in tqdm(test_data_norm):
    # Similarly vectors...electra_extracontext and vectors...greberta
    # If using text embeddings, use_text_vectors=True
    classifier = wsd.train_lemma_model(lemma,training_data,test_data_norm,'wsd/vectors_train_electra','wsd/vectors_test/vectors_test_electra.pickle',use_text_vectors=False)
    all_predictions.update(classifier.predictions)

In [ ]:
wsd.get_accuracy_by_majority_sense(training_data,test_data_norm,all_predictions)

In [ ]:
wsd.get_accuracy_by_num_senses(training_data,test_data_norm,all_predictions)

In [ ]:
def get_synset_definition(synset):
    definition = f'{", ".join(synset.lemma_names())}: {synset.definition()} "{"/".join(synset.examples())}"'
    if definition.endswith(' ""'):
        definition = definition[:-3]
    return definition

In [ ]:
from nltk.corpus import wordnet as wn
with open('wsd/results_analysis.tsv','w',encoding='utf8') as outfile:
    lemma_senses = {}
    for lemma, tokens in training_data.items():
        lemma_senses[lemma] = len(set(tokens.values()))
    lemma_baseline = wsd.get_baseline(training_data)
    for lemma, tokens in test_data_norm.items():
        for token in tokens.keys():
            correct = all_predictions[token] in test_data_norm[lemma][token].split('|')
            majority_sense = lemma_baseline[lemma] in test_data_norm[lemma][token].split('|')
            definition_prediction = get_synset_definition(wn.synset(all_predictions[token]))
            definition_gold = '|'.join([get_synset_definition(wn.synset(x)) for x in test_data_norm[lemma][token].split('|')])
            outfile.write(f'{token}\t{lemma}\t{'|'.join(gold_synset[token])}\t{test_data_norm[lemma][token]}\t{all_predictions[token]}\t{correct}\t{definition_gold}\t{definition_prediction}\t{lemma_senses[lemma]}\t{majority_sense}\n')